In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_0.pickle

trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['question_of_interest_cell23']
me:  11
trying: ['count_then_return_percent_23']
me:  11
trying: ['percentages_cell23']
me:  11
trying: ['orient

me:  17
trying: ['df']
me:  14
trying: ['responses_df_2021']
me:  17
trying: ['count_then_return_percent_33']
me:  21
trying: ['px']
me:  0
trying: ['responses_df_2019_cell10']


me:  21
trying: ['warnings']
me:  0
trying: ['np']
me:  0
trying: ['responses_df_2020']
me:  21
trying: ['go']
me:  0
trying: ['subset_of_countries']
me:  6
trying: ['question_of_interest_cell33']
me:  21
trying: ['responses_df_2018_cell10']
me:  21
trying: ['country_df_combined']
me:  6
trying: ['title_for_x_axis']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_33']
me:  21
trying: ['question_name_alternate']
me:  6
trying: ['programming_df_combined']
me:  21
trying: ['question_name_alternate_cell18']
me:  6
trying: ['count_then_return_percent_18']
me:  6
trying: ['add_year_column_to_dataframes_18']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  6
trying: ['country_df_combined_cell18']
me:  6
trying: ['question_name']
me:  6
trying: ['question_of_interest_alternate']
me:  14
trying: ['question_of_interest_cell18']
me:  6
trying: ['question_of_interest_cell26']
me:  14
trying: ['grab_subset_of_data_26']
me:  14
trying: ['add_year_column_to_dataframes

In [3]:
%%RecordEvent
%%time
### cell 23 ###

def grab_subset_of_data_35(df, question_of_interest):
    # Vectorized column filtering and renaming without an explicit copy
    mask = df.columns.str.contains(question_of_interest)
    cols = df.columns[mask]
    new_names = cols.str.rsplit('-', 1).str[-1].str.lstrip()
    df_sub = df.loc[:, cols]
    df_sub.columns = new_names
    return df_sub.dropna(how='all', subset=new_names)


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(question_of_interest, include_2017=None):
    # Define (dataframe, year) pairs in chronological order
    sources = [
        (responses_df_2018_cell10, '2018'),
        (responses_df_2019_cell10, '2019'),
        (responses_df_2020,        '2020'),
        (responses_df_2021,        '2021'),
        (responses_df_2022_cell10, '2022'),
    ]
    if include_2017 is not None:
        sources.insert(0, (responses_df_2017, '2017'))
    # Subset, rename, dropna, and assign year in one comprehension
    dfs = [
        grab_subset_of_data_35(src_df, question_of_interest).assign(year=yr)
        for src_df, yr in sources
    ]
    # Concatenate once with a fresh integer index
    df_combined = pd.concat(dfs, ignore_index=True)
    # Count non-null responses per year
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # Cast counts to int and set 'year' as index for alignment
    df_counts_int = df_counts.astype(int).set_index('year')
    # Compute total respondents per year once
    totals = df['year'].value_counts().reindex(df_counts_int.index)
    # Divide and multiply by 100, then bring 'year' back as a column
    percentages = df_counts_int.div(totals, axis=0).mul(100).reset_index()
    return percentages

# ---- Pipeline ----
question_of_interest_cell35 = 'What programming languages do you use on a regular basis?'

language_df_combined, language_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )

language_df_combined_percentages = \
    convert_df_of_counts_to_percentages_35(
        language_df_combined,
        language_df_combined_counts
    )

# Select the languages of interest and reshape
languages = ['Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']

df_cell35 = (
    language_df_combined_percentages
        .loc[:, ['year'] + languages]
        .melt(id_vars=['year'], value_vars=languages)
        .sort_values(by=['year', 'value'])
        .rename(columns={'variable': ''})
)

df_cell35.info()

TypeError: StringMethods.rsplit() takes from 1 to 2 positional arguments but 3 were given

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_23_try_0.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_23_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[23], f)


In [ ]:
opt_output = Out.get(4)